<a href="https://colab.research.google.com/github/ridoy1211/Flyrank-Internship-ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ridoy1211/Flyrank-Internship-ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os

if not os.path.exists('data/raw/content_refresh_anonymized.csv'):
    if not os.path.exists('Flyrank-Internship-ML'):
        !git clone https://github.com/ridoy1211/Flyrank-Internship-ML.git
    os.chdir('Flyrank-Internship-ML')

print("Working directory:", os.getcwd())
print("File exists:", os.path.exists('data/raw/content_refresh_anonymized.csv'))

Cloning into 'Flyrank-Internship-ML'...
remote: Enumerating objects: 132, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 132 (delta 44), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (132/132), 1.85 MiB | 4.53 MiB/s, done.
Resolving deltas: 100% (44/44), done.
Working directory: /content/Flyrank-Internship-ML
File exists: True


## 1. My lane as an ML task (type)

**Task type: Classification (binary), with the output used to build a ranked list.**

The core task is binary classification: for each content page, predict whether it shows the
"Great Decoupling" signature — impressions holding steady while clicks meaningfully drop.
That's a yes/no label per page, which makes it classification, not clustering (I'm not looking
for unlabeled groupings) and not pure ranking (I do have a defined positive class, not just a
relative ordering with no ground truth).

That said, the *output a strategist actually uses* is a ranked review queue, not a raw label —
exactly like the starter pipeline's own approach (`baseline_refresh_score` → classification
probability → `final_refresh_score` used to rank). So the shape is: classify first, then use the
predicted probability to rank pages for review. I'm naming it classification because that's the
thing the model is actually trained to do; ranking is what I do with its output afterward.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
pass


## 2. Target or proxy

**Target (a defined-rule proxy, not yet a validated future outcome):**

decoupling_signature = 1 if:
    impressions changed between -10% and +10% from the prior 30-day window to the last 30-day window
    AND clicks dropped 15% or more over that same comparison
else 0

This comes from a rule **I defined myself**, not an observed future outcome and not a FlyRank
product flag (FlyRank's product decisions were never shipped in this data in the first place). It
mirrors the "SERP or AI click loss" look-alike pattern named in the lane guide's section 7,
operationalized using the `_prev_30d` / `_last_30d` columns that already exist in the starter
data — a genuine before/after comparison, not a single-window snapshot.

**Being honest about what kind of proxy this is:** both windows compared here (`prev_30d` and
`last_30d`) are already in the past relative to the dataset snapshot — so right now this is a
*detection* label (did this pattern already happen), not a *forecast* label (will it happen
next). A stronger, future-looking version — prior 60-90 days of features predicting a decoupling
signature in the *next* 30 days — is the natural next step once I get to leakage-audit work in a
later week; I'm naming that gap now rather than quietly treating today's proxy as more than it is.

In [3]:
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

sub = df[(df['impressions_prev_30d'] > 0) & (df['clicks_prev_30d'] > 0)].copy()

sub['impr_change_pct'] = 100 * (sub['impressions_last_30d'] - sub['impressions_prev_30d']) / sub['impressions_prev_30d']
sub['click_change_pct'] = 100 * (sub['clicks_last_30d'] - sub['clicks_prev_30d']) / sub['clicks_prev_30d']

sub['decoupling_signature'] = (
    sub['impr_change_pct'].between(-10, 10) & (sub['click_change_pct'] <= -15)
).astype(int)

print(f"usable rows: {len(sub)}")
print(f"positive rate: {sub['decoupling_signature'].mean():.4f} "
      f"({sub['decoupling_signature'].sum()} of {len(sub)} rows)")


usable rows: 11759
positive rate: 0.0573 (674 of 11759 rows)


## 3. Success metric

**Primary metric: Precision@50** (matching the starter pipeline's own reporting, so results are
comparable), with **Average Precision** as a secondary check across the whole ranking.

**Why not plain accuracy:** the positive rate is about 5.7% (computed below). A model that
predicts "0" for every page gets ~94% accuracy while catching zero real cases — accuracy would
reward doing nothing. Precision@K matches how the output actually gets used: a strategist reviews
a fixed number of pages per cycle (the starter pipeline uses top 50), so "of the top 50 pages the
model flags, how many are real" is the number that actually maps to their real capacity and real
cost of a wrong call.

**What "good" means concretely:** the starter baseline rule alone scores 0.240 precision@50 on
the (different) decline label — beating a comparable fixed-rule baseline on *this* label, by a
meaningful margin, is the bar for saying a learned model earned its place over just hard-coding
the rule above.

In [4]:
positive_rate = sub['decoupling_signature'].mean()
naive_accuracy_if_predict_all_zero = 1 - positive_rate

print(f"positive rate: {positive_rate:.4f}")
print(f"accuracy from just predicting 'not decoupling' every time: {naive_accuracy_if_predict_all_zero:.4f}")
print("-> confirms accuracy is a bad metric here; precision@50 / average precision are used instead.")


positive rate: 0.0573
accuracy from just predicting 'not decoupling' every time: 0.9427
-> confirms accuracy is a bad metric here; precision@50 / average precision are used instead.


## 4. The unit of analysis, as a real dataframe

One row = **one content page (`content_id`), evaluated using its prior-30-day vs. last-30-day
windows** — the same grain as the target definition above. Showing the actual columns a model
would see, plus the label, below.

In [5]:
analysis_cols = [
    'content_id', 'client_id', 'content_type', 'main_intent', 'position_tier',
    'impressions_prev_30d', 'impressions_last_30d', 'impr_change_pct',
    'clicks_prev_30d', 'clicks_last_30d', 'click_change_pct',
    'decoupling_signature',
]

unit_of_analysis_df = sub[analysis_cols].copy()
print(f"shape: {unit_of_analysis_df.shape}  (one row per content page)")
unit_of_analysis_df.sort_values('decoupling_signature', ascending=False).head(8)


shape: (11759, 12)  (one row per content page)


,content_id,client_id,content_type,main_intent,position_tier,impressions_prev_30d,impressions_last_30d,impr_change_pct,clicks_prev_30d,clicks_last_30d,click_change_pct,decoupling_signature
10681,content_8642a961dd74,client_8b940be7fb,keyword article,informational,page_1,1977,1895,-4.147699,21,16,-23.809524,1
24962,content_3a7deadb6be0,client_19581e27de,keyword article,informational,deep,120,132,10.000000,1,0,-100.000000,1
23489,content_b6f4567039ed,client_19581e27de,keyword article,transactional,page_1,12029,10961,-8.878544,42,34,-19.047619,1
1623,content_005f9cfd8d8f,client_bbb965ab0c,keyword article,informational,page_1,2152,2174,1.022305,11,6,-45.454545,1
10673,content_941f15099b91,client_e629fa6598,keyword article,transactional,top_3,1006,995,-1.093439,6,3,-50.000000,1
16712,content_271940834e91,client_4e07408562,keyword article,commercial,page_1,7695,7550,-1.884340,29,15,-48.275862,1
16715,content_510559cd6193,client_3fdba35f04,keyword article,informational,striking,151,149,-1.324503,1,0,-100.000000,1
24952,content_e8f2b01c3603,client_f369cb89fc,keyword article,commercial,striking,72,69,-4.166667,1,0,-100.000000,1


## 5. Why ML beats a fixed rule here

The rule in Section 2 already *detects* the pattern — so why not just ship that rule as the whole
system? Because the rate at which pages fall into this pattern is not uniform, and a single global
threshold treats every page the same when the real risk clearly isn't the same:

- **By position tier:** pages ranked "deep" show the signature at ~12.7% vs. ~4.9%-6.3% for
other tiers — over 2x the base rate, computed below.
- **By content type:** "comparison article" pages show it at 15.0% vs. 5.6% for "keyword
article" pages, the dominant content type in this data.

A fixed if/then rule can only either (a) apply the same threshold to everyone, missing that some
segments are structurally much riskier, or (b) turn into a sprawling pile of manually-tuned
per-segment thresholds that gets unmaintainable fast and still can't capture interactions between
position, content type, freshness, and intent at once. That's exactly the kind of multivariate,
interacting pattern a model can learn jointly instead of a human hand-tuning dozens of thresholds —
which is the actual argument for ML here, not just "ML is more advanced."

In [6]:
print('Decoupling rate by position_tier:')
print(sub.groupby('position_tier')['decoupling_signature'].agg(['mean', 'count']).round(4))
print()

top_types = sub['content_type'].value_counts().head(3).index
print('Decoupling rate by content_type (top 3 by volume):')
print(sub[sub['content_type'].isin(top_types)].groupby('content_type')['decoupling_signature'].agg(['mean', 'count']).round(4))


Decoupling rate by position_tier:
                 mean  count
position_tier               
deep           0.1268     71
page_1         0.0631   5987
page_3_5       0.0485   2390
striking       0.0507   2959
top_3          0.0597    352

Decoupling rate by content_type (top 3 by volume):
                      mean  count
content_type                     
comparison article  0.1500     40
feedly article      0.1029    243
keyword article     0.0560  11476


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.